# neuralCAD-Edit

neuralCAD-Edit is stored as a mongita database containing metadata and filepaths, which point to objects (images, .step etc.) in the file tree. 

This notebook contains some ways to visualise the data. We recommend always accessing the data through the database, and the access patterns used here should give you and idea about how to do this.

### Imports and config

Make sure the paths in the config are correct

In [ ]:
import os
import os.path as osp
import sys
module_path = os.path.abspath(os.path.join('../..'))
sys.path.append(module_path)
from src.utils.db import DatabaseManager
from src.utils.process_config import load_config
from IPython.display import display
from PIL import Image as PILImage, ImageDraw, ImageFont
import shutil

config_path = os.path.join(module_path, 'src', 'config', 'edit_192_external.json')

class Args:
    def __init__(self, config):
        self.config = config
args = Args(config=config_path)
config = load_config(args.config)
dbm = DatabaseManager(config)
print("Database manager initialized from config:", config_path)
dbm.print_db_schema_counts()


### Function defs

Just run. You can collapse to get it out of the way.

In [ ]:
def display_images_side_by_side(image_paths, new_height=512, labels=None, n_horizontal=4):
    """
    Display a list of local image paths side by side with optional labels above each image.
    If more images are provided than the n_horizontal limit, they will wrap to the next line.

    Args:
        image_paths (list): List of file paths to images
        new_height (int): Height to resize images to
        labels (list, optional): List of text labels to display above each image
        n_horizontal (int): Maximum number of images per row
    """
    
    # Load and resize images
    images = []
    for img_path in image_paths:
        if os.path.exists(img_path):
            try:
                img = PILImage.open(img_path)
                # Calculate new width to maintain aspect ratio
                width_ratio = new_height / img.height
                new_width = int(img.width * width_ratio)
                img = img.resize((new_width, new_height))
                images.append(img)
            except Exception as e:
                print(f"Error loading image {img_path}: {e}")
    
    if not images:
        print("No valid images found to display")
        return
    
    # Calculate grid dimensions
    n_rows = (len(images) + n_horizontal - 1) // n_horizontal  # Ceiling division
    
    # Group images into rows
    image_rows = []
    label_rows = []
    
    for row_idx in range(n_rows):
        start_idx = row_idx * n_horizontal
        end_idx = min(start_idx + n_horizontal, len(images))
        
        row_images = images[start_idx:end_idx]
        row_labels = labels[start_idx:end_idx] if labels else None
        
        image_rows.append(row_images)
        label_rows.append(row_labels)
    
    # Calculate dimensions for the combined image
    label_height = 30 if labels else 0
    row_height = new_height + label_height
    
    # Find the maximum width needed (widest row)
    max_row_width = 0
    for row_images in image_rows:
        row_width = sum(img.width for img in row_images)
        max_row_width = max(max_row_width, row_width)
    
    total_height = row_height * n_rows
    
    # Create the combined image
    combined_image = PILImage.new('RGB', (max_row_width, total_height), color=(255, 255, 255))
    draw = ImageDraw.Draw(combined_image)
    
    try:
        font = ImageFont.truetype("Arial", 14)
    except IOError:
        font = ImageFont.load_default()
    
    # Process each row
    for row_idx, (row_images, row_labels) in enumerate(zip(image_rows, label_rows)):
        current_y = row_idx * row_height
        current_x = 0
        
        for i, img in enumerate(row_images):
            # Paste image
            combined_image.paste(img, (current_x, current_y + label_height))
            
            # Add label if provided
            if row_labels and i < len(row_labels):
                text = str(row_labels[i])
                text_width = draw.textlength(text, font=font)
                text_x = current_x + (img.width - text_width) // 2
                text_y = current_y + 5
                draw.text((text_x, text_y), text, fill=(0, 0, 0), font=font)
            
            current_x += img.width
    
    # Display the combined image
    display(combined_image)

def display_request_info(dbm, request_id):
    request = dbm.requests.find_one({"_id": request_id})
    if request_id is None:
        print(f"No request found with ID: {request_id}")
        return
    
    print("Request details:")
    print(f"Request ID: {request_id}")
    print(f"Keys: {list(request.keys())}")

    print()
    print("Brep:")
    breps = dbm.breps.find_one({"_id": request["brep_start"]})
    for key, value in breps.items():
        if value:
            if len(str(value)) > 500:
                value = '...'
            print(f"{key}: {value}")

    print()
    print("User details:")
    user = dbm.users.find_one({"_id": request["user"]})
    for key, value in user.items():
        if value:
            print(f"{key}: {value}")

    # show the instruction text
    print()
    instruction_text = request.get("text")
    if instruction_text:
        print("Instruction text:")
        print(instruction_text)
    else:
        print("No instruction text found for this request.")

def display_edit_info(dbm, edit_id, n_to_display=4):
    edit = dbm.edits.find_one({"_id": edit_id})
    if edit_id is None:
        print(f"No edit found with ID: {edit_id}")
        return
    print("Edit details:")
    print(f"Edit ID: {edit_id}")
    print(f"Keys: {list(edit.keys())}")

    print()
    print("Events")
    print(f"Number of events: {len(edit['events'])}")
    # pick n_to_display uniformly from the events
    if len(edit['events']) > n_to_display:
        step = len(edit['events']) // n_to_display
        events_to_display = edit['events'][::step][:n_to_display]
    else:
        events_to_display = edit['events']
    for event in events_to_display:
        print(event)

    print()
    print("brep end:")
    breps = dbm.breps.find_one({"_id": edit["brep_end"]})
    for key, value in breps.items():
        if value:
            if len(str(value)) > 500:
                value = '...'
            print(f"{key}: {value}")

    # show the toprightiso image of the brep end
    brep_images = dbm.get_brep_images(edit["brep_end"], views=["toprightiso"], format=["jpg", "png"])
    if brep_images:
        brep_image_path = os.path.join(dbm.root_dir, brep_images[0])
        display_images_side_by_side([brep_image_path])

    print()
    print("Frames:")
    frames_dir = os.path.join(dbm.root_dir, edit["frames_dir"])
    if os.path.exists(frames_dir):
        print(f"Frames directory found at: {frames_dir}")
        frames = os.listdir(frames_dir)
        frames.sort()
        display_images_side_by_side([os.path.join(frames_dir, f) for f in frames])
    else:
        print(f"Frames directory not found at: {frames_dir}")

def display_request_and_edits(dbm, request_id, output_dir=None, views=["toprightiso", "sketch", "render"], n_horizontal=4, height=384, models=[]):
    request = dbm.requests.find_one({"_id": request_id})
    edits = dbm.edits.find({"request": request_id})

    request_id = request["_id"]
    print(f"Request ID: {request_id}")
    user_2_img = {}
    for edit in edits:
        brep = dbm.breps.find_one({"_id": edit["brep_end"]})
        brep_id = brep["_id"]
        brep_img = dbm.get_brep_images(brep_id, views=views)
        if brep_img:
            brep_img = brep_img[0]
        else:
            continue
        brep_img = os.path.join(dbm.root_dir, brep_img)

        user = dbm.users.find_one({"_id": edit["user"]})

        if edit["user"] == request["user"]:
            display_user = "Human GT"
            # user_2_img["Human GT"] = brep_img
        elif user["is_human"]:
            display_user = "Human Other"
            # user_2_img["Human Other"] = brep_img
        else:
            display_user = user["_id"]
            # user_2_img[user["_id"]] = brep_img
        
        if len(models) > 0 and display_user not in models:
            continue


        user_2_img[display_user] = brep_img
        print(f"Edit ID: {edit['_id']}, User: {user['_id']}, BRep Image: {brep_img}")

    # display the request instruction text
    instruction_text = request.get("text")
    if instruction_text:
        print("Instruction text:")
        print(instruction_text)

    # display the request brep images
    brep_start = dbm.breps.find_one({"_id": request["brep_start"]})
    if brep_start:
        request_images = dbm.get_brep_images(brep_start["_id"], views=views, format=["jpg", "png"])
        
        request_images = [os.path.join(dbm.root_dir, img) for img in request_images]
        display_images_side_by_side(
            request_images,
            labels=[""] * len(request_images),
            new_height=height
        )
        if output_dir:
            for idx, img_path in enumerate(request_images):
                basename = os.path.basename(img_path)
                output_folder = os.path.join(output_dir, request_id)
                os.makedirs(output_folder, exist_ok=True)
                output_path = os.path.join(output_folder, basename)
                print(img_path, "->", output_path)
                shutil.copy(img_path, output_path)

    # sort the user_2_img dictionary by user name
    user_2_img = dict(sorted(user_2_img.items(), key=lambda item: item[0]))


    # if a list of models is provided, filter the user_2_img to only include those models (if they exist)
    if models:
        user_2_img = {user: img for user, img in user_2_img.items() if user in models}



    labels, images = [], []
    for user, img in user_2_img.items():
        labels.append(user)
        images.append(img)
    display_images_side_by_side(images, labels=labels, n_horizontal=n_horizontal, new_height=height)
    if output_dir:
        for img_path, user in zip(images, labels):
            basename = os.path.basename(img_path)
            output_folder = os.path.join(output_dir, request_id, user)
            os.makedirs(output_folder, exist_ok=True)
            output_path = os.path.join(output_folder, basename)
            print(img_path, "->", output_path)
            shutil.copy(img_path, output_path)


### Display all information about a single request
Here, we find a request from the database, which is from an medium difficuly draw-static request on a parametric assembly task.


In [ ]:
request_ids = dbm.requests.find({"modality": "text", "difficulty": "medium", "assembly": True, "parametric": True})
request_ids = [request["_id"] for request in request_ids]

display_request_info(dbm, request_ids[0] if request_ids else None)

### Display all information about a single edit

Here, we take a request, and find an edit which corresponds to that request.

In [ ]:
request_ids = dbm.requests.find({"modality": "text", "difficulty": "medium", "assembly": True, "parametric": True})
request_ids = [request["_id"] for request in request_ids]

edit_ids = dbm.edits.find({"request": {"$in": request_ids}})
edit_ids = [edit["_id"] for edit in edit_ids]

display_edit_info(dbm, edit_ids[0])

### Display all requests and the output of different users/AIs

Requests are shown below. If an AI did not produce a valid CAD model, then nothing is shown.

In [ ]:
# Load request ids from the database (adjust the query to filter as needed)
request_query = {"request_type": "edit"}
request_ids = [request["_id"] for request in dbm.requests.find(request_query)]
request_ids.sort()
print(f"Found {len(request_ids)} requests")

models = [
    "Human GT",
    "Human Other",
    'gemini-3-pro_cadquery-script',
    "claude-sonnet-4.5_cadquery-script",
    "gpt-5.2_cadquery-script",
]

for request_id in request_ids:
    display_request_and_edits(dbm, request_id, output_dir=None, views=["dslrender","sketch", "render", "toprightiso"], n_horizontal=5, height=512, models=models)

In [ ]:
# Test requests.

request_ids = ['SUJ2G2UMJQR7PMBX_1759209987.785593',
'3YH2WFSRM22W7DKT_1769773335.525203',
'B7A2N74ZJBF9MZHU_1770174133.012106',
'F332D3FXML85WLR2_1769607142.566352',
'ZK22J6VYRKQ2RTFD_1758874422.1403751',
]

models = [
    "Human GT",
    "Human Other",
    'gemini-3-pro_cadquery-script',
    "claude-sonnet-4.5_cadquery-script",
    "gpt-5.2_cadquery-script",
]

for request_id in request_ids:
    display_request_and_edits(dbm, request_id, output_dir=None, views=["dslrender","sketch", "render", "toprightiso"], n_horizontal=5, height=512, models=models)